# Corr of PMID_35561311 and ASO data  
gene exp matrix & GSEA NES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from adjustText import adjust_text
import matplotlib as mpl
from matplotlib import font_manager

arial_path = "/media/scratch/fy2306/tools/fonts"
font_files = font_manager.findSystemFonts(fontpaths=arial_path)

for file in font_files:
    font_manager.fontManager.addfont(file)
    
mpl.rcParams['font.family'] = 'Arial'

In [ ]:
def compare_correlation(df1_path, df2_path, col_name, match_by, plot_title, df1_name, df2_name, filter_column = None, filter_threshold = None, annotate = False, save = False):

	df1 = pd.read_csv(df1_path, sep="\t")
	df2 = pd.read_csv(df2_path, sep="\t")

	merged = pd.merge(df1, df2, on=match_by, suffixes=('_df1', '_df2'))

	merged = merged.dropna(subset=[f"{col_name}_df1", f"{col_name}_df2"])
	if filter_column is not None:
		merged = merged[(merged[f"{filter_column}_df1"] < filter_threshold) & (merged[f"{filter_column}_df2"] < filter_threshold)]
	print(merged.head(15))

	x = merged[f"{col_name}_df1"]
	y = merged[f"{col_name}_df2"]
	labels = merged[match_by]

	pearson_r, _ = pearsonr(x, y)
	spearman_r, _ = spearmanr(x, y)
	n = len(merged)

	if annotate:
		plt.figure(figsize=(6, 4))
		plt.scatter(x, y, c='#AA66FF', alpha=0.8, s=15)
	else:
		plt.figure(figsize=(4, 4))
		plt.scatter(x, y, c='#AA66FF', alpha=0.6, s=5)
	plt.xlabel(df1_name, fontsize=12)
	plt.ylabel(df2_name, fontsize=12)

	text_str = f"Pearson r = {pearson_r:.3f}\nSpearman r = {spearman_r:.3f}"
	plt.text(0.05, 0.95, text_str, transform=plt.gca().transAxes,
				fontsize=12, verticalalignment='top')
	
	ax = plt.gca()
	if annotate:
		xmax = x.max()
		xmin = x.min()
		x_range = xmax - xmin
		x_offset = x_range * 0.05
		x_text = xmax + x_range * 0.15

		data_for_labels = sorted(zip(x, y, labels), key=lambda v: v[1])
		n_labels = len(data_for_labels)

		y_label_positions = np.linspace(min(y), max(y), n_labels)

		for (xi, yi, label), y_text in zip(data_for_labels, y_label_positions):
			ax.plot([xi, x_text], [yi, y_text],
					lw=0.5, color='gray', alpha=0.7)

			ax.text(
				x_text, y_text, label.split("_", 1)[1],
				va='center', ha='left',
				fontsize=11, color='black'
			)

		ax.set_xlim(xmin - x_offset, x_text + x_offset)

	plt.title(f"{plot_title}\n(N={n})", fontsize=13)
	plt.tick_params(axis='both', labelsize=12)
	plt.grid(False)
	ax = plt.gca()
	ax.spines['top'].set_visible(False)
	ax.spines['right'].set_visible(False)
	plt.tight_layout()
	if save:
		plt.savefig("/media/scratch/fy2306/projects/base_editing/plots/ASO/GSEA.wSTING.pdf", 
					bbox_inches="tight",
					dpi=300,              
					transparent=True,
					format='pdf')
	plt.show()

In [ ]:
compare_correlation(
	df1_path = "/media/scratch/fy2306/projects/base_editing/results/ASO_rnaseq_batch_2_outliers/de_deseq2.48h.tsv", 
	df2_path = "/media/scratch/fy2306/projects/base_editing/results/PMID_35561311/de_deseq2.tsv", 
	col_name = "log2FoldChange", 
	match_by = "gene_id", 
	plot_title = "Gene log2FoldChange", 
	df1_name = "ASO 48 h Treatment vs Control", 
	df2_name = "PMID_35561311 STING vs GFP")
compare_correlation(
	df1_path = "/media/scratch/fy2306/projects/base_editing/results/ASO_rnaseq_batch_2_outliers/de_deseq2.48h.tsv", 
	df2_path = "/media/scratch/fy2306/projects/base_editing/results/PMID_35561311/de_deseq2.tsv", 
	col_name = "log2FoldChange", 
	match_by = "gene_id", 
	plot_title = "Gene log2FoldChange (both padj < 0.05)", 
	df1_name = "ASO 48 h Treatment vs Control", 
	df2_name = "PMID_35561311 STING vs GFP",
	filter_column = "padj",
	filter_threshold = 0.05)
compare_correlation(
	df1_path = "/media/scratch/fy2306/projects/base_editing/results/ASO_rnaseq_batch_2_outliers/enrichment_48h/gsea_msig_H_std.tsv", 
	df2_path = "/media/scratch/fy2306/projects/base_editing/results/PMID_35561311/gsea_msig_H_std.tsv", 
	col_name = "NES", 
	match_by = "ID", 
	plot_title = "Hallmark gene sets", 
	df1_name = "ASO 48 h Treatment vs Control", 
	df2_name = "PMID_35561311 STING vs GFP")
compare_correlation(
	df1_path = "/media/scratch/fy2306/projects/base_editing/results/ASO_rnaseq_batch_2_outliers/enrichment_48h/gsea_msig_H_std.tsv", 
	df2_path = "/media/scratch/fy2306/projects/base_editing/results/PMID_35561311/gsea_msig_H_std.tsv", 
	col_name = "NES", 
	match_by = "ID", 
	plot_title = "Hallmark gene sets (both p.adjust < 0.05)", 
	df1_name = "ASO 48 h Treatment vs Control", 
	df2_name = "PMID_35561311 STING vs GFP",
	filter_column = "p.adjust",
	filter_threshold = 0.05,
	annotate = True)

In [ ]:
compare_correlation(
	df1_path = "/media/scratch/fy2306/projects/base_editing/results/ASO_rnaseq_batch_2_outliers/de_deseq2.72h.tsv", 
	df2_path = "/media/scratch/fy2306/projects/base_editing/results/PMID_35561311/de_deseq2.tsv", 
	col_name = "log2FoldChange", 
	match_by = "gene_id", 
	plot_title = "Gene log2FoldChange", 
	df1_name = "ASO 72 h Treatment vs Control", 
	df2_name = "PMID_35561311 STING vs GFP")
compare_correlation(
	df1_path = "/media/scratch/fy2306/projects/base_editing/results/ASO_rnaseq_batch_2_outliers/de_deseq2.72h.tsv", 
	df2_path = "/media/scratch/fy2306/projects/base_editing/results/PMID_35561311/de_deseq2.tsv", 
	col_name = "log2FoldChange", 
	match_by = "gene_id", 
	plot_title = "Gene log2FoldChange (both padj < 0.05)", 
	df1_name = "ASO 72 h Treatment vs Control", 
	df2_name = "PMID_35561311 STING vs GFP",
	filter_column = "padj",
	filter_threshold = 0.05)
compare_correlation(
	df1_path = "/media/scratch/fy2306/projects/base_editing/results/ASO_rnaseq_batch_2_outliers/enrichment_72h/gsea_msig_H_std.tsv", 
	df2_path = "/media/scratch/fy2306/projects/base_editing/results/PMID_35561311/gsea_msig_H_std.tsv", 
	col_name = "NES", 
	match_by = "ID", 
	plot_title = "Hallmark gene sets", 
	df1_name = "ASO 72 h Treatment vs Control", 
	df2_name = "PMID_35561311 STING vs GFP")
compare_correlation(
	df1_path = "/media/scratch/fy2306/projects/base_editing/results/ASO_rnaseq_batch_2_outliers/enrichment_72h/gsea_msig_H_std.tsv", 
	df2_path = "/media/scratch/fy2306/projects/base_editing/results/PMID_35561311/gsea_msig_H_std.tsv", 
	col_name = "NES", 
	match_by = "ID", 
	plot_title = "Hallmark gene sets (both p.adjust < 0.05)", 
	df1_name = "ASO 72 h Treatment vs Control", 
	df2_name = "PMID_35561311 STING vs GFP",
	filter_column = "p.adjust",
	filter_threshold = 0.05,
	annotate = True,
	save = True)